# Tech Challenge IADT Fase 4 - Sistema Multimodal Completo

# Apresentacao: 15 minutos

## Objetivo

Este projeto implementa um **sistema de análise multimodal** para monitoramento médico, conforme o Tech Challenge da Fase 4.
Ele integra três fontes de dados e as combina em uma única decisão clínica:

- **Vídeo**: postura com **OpenPose‑Light (MediaPipe)** + objetos/áreas críticas com **YOLOv8**.
- **Áudio** (consultas médicas): transcrição e análise de sentimento/entidades com **Azure Cognitive Services** (Speech to Text + Text Analytics), com detecção de alterações vocais (fadiga, falta de ar, dor).
- **Anomalias**: séries temporais de sinais vitais, evolução de prescrições e padrões de movimentação.

Os três módulos alimentam um **workflow integrado** que calcula uma **pontuação de risco** e gera o **alerta final à equipe médica**. O notebook roda em dois modos (escolhidos na execução): **offline** (dados simulados, sem chaves) ou **azure** (serviços reais).

## Seção 0 — Preparação do ambiente e modo de execução

Carrega as bibliotecas de visão computacional e dados e define em qual
modo o sistema vai rodar (offline para demonstração ou Azure real).
Também inicializa o MediaPipe Pose (OpenPose-Light) para análise postural.

In [ ]:
import cv2          # leitura/manipulação de vídeo/frames
import numpy as np  # operações numéricas
import torch        # YOLOv8
import mediapipe as mp  # OpenPose-Light: análise postural
import pandas as pd # séries temporais
from scipy import stats # estatísticas para detecção de anomalias
import os

# Switch de execução: MODO = "offline" (sem Azure) ou "azure" (serviços reais)
try:
    MODO = input("Modo de execucao [offline/azure] (Enter = offline): "
                 ).strip().lower() or "offline"
except Exception:
    MODO = os.getenv("MODO", "offline")
CAMINHO_VIDEO = os.getenv("CAMINHO_VIDEO", "dados/video.mp4")
CAMINHO_AUDIO = os.getenv("CAMINHO_AUDIO", "dados/audio.wav")
AZURE_ON = (MODO == "azure")
print("MODO de execucao:", MODO)

# MediaPipe Pose (OpenPose-Light)
try:
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    _mp_ok = True
except Exception:
    pose = None
    _mp_ok = False

## Log de execução (`logsetup`)

Configura o logging para gravar em arquivo (`logs/execucao_<timestamp>.log`)
e no console, e cria o `REGISTRO_API` — uma lista que acumula o retorno
**bruto de cada chamada/API/passo** para auditoria. Funções auxíliares:
`_resumir(dado)` (trunca textos longos) e `registrar_api(nome, dado)`
(acumula e imprime no log).

In [ ]:
import logging, datetime, json as _json

LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR,
                        "execucao_%s.log"
                        % datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, encoding="utf-8"),
              logging.StreamHandler()],
)
log = logging.getLogger("techchallenge")
log.info("===== INICIO DA EXECUCAO | log em: %s =====", LOG_FILE)

REGISTRO_API = []  # [(nome, dado)] de cada chamada/passo

def _resumir(dado, limite=400):
    try:
        t = _json.dumps(dado, ensure_ascii=False, default=str)
    except Exception:
        t = str(dado)
    return t if len(t) <= limite else t[:limite] + "...(+%d chars)" % (
        len(t) - limite)

def registrar_api(nome, dado):
    REGISTRO_API.append({"nome": nome, "dado": dado})
    log.info("[RETORNO] %s -> %s", nome, _resumir(dado))
    return dado

In [ ]:
# ---------------------------------------------------------------------
# Modelo YOLOv8 — detecção de objetos e áreas críticas no vídeo clínico.
# Na 1ª execução os pesos 'yolov8n.pt' são baixados e depois ficam em cache.
# Se o download falhar (ex.: sem internet), seguimos em modo simulado.
# ---------------------------------------------------------------------
try:
    from ultralytics import YOLO
    model_yolo = YOLO('yolov8n.pt')
except Exception as e:
    print("YOLO indisponivel -> usarei objetos simulados:", e)
    model_yolo = None
_yolo_online = model_yolo is not None  # flag que controla real x simulado
print("YOLO disponivel:", _yolo_online)

## `detectar_objetos_yolo(frame)` — detecção de objetos no frame

Recebe um frame (imagem) e devolve a lista de objetos detectados pelo YOLOv8.

**Entrada:** `frame` (imagem BGR do OpenCV).  
**Saída:** lista de dicionários com `label`, `confidence` e `bbox` `[x1,y1,x2,y2]`.

**Lógica:** se o modelo estiver carregado (`_yolo_online`), roda
`model_yolo(frame)` e percorre `results[0].boxes`, extraindo cada detecção
(coordenadas, confiança e classe COCO). Se o YOLO não estiver disponível
(sem download/internet), retorna objetos simulados para a demonstração não travar.

In [ ]:
def detectar_objetos_yolo(frame):
    if _yolo_online and model_yolo is not None:
        resultados = model_yolo(frame)
        objetos = []
        caixas = resultados[0].boxes
        for caixa in caixas:
            x1, y1, x2, y2 = caixa.xyxy[0].tolist()
            conf = float(caixa.conf[0])
            cls = int(caixa.cls[0])
            label = model_yolo.names[cls]
            objetos.append({'label': label, 'confidence': conf,
                            'bbox': [int(x1), int(y1), int(x2), int(y2)]})
        return objetos
    else:
        obj1 = {'label': 'instrumento_critico', 'confidence': 0.85,
                'bbox': [100, 100, 200, 300]}
        obj2 = {'label': 'area_perigosa', 'confidence': 0.72,
                'bbox': [300, 50, 400, 200]}
        return [obj1, obj2]

## `detectar_postura(frame)` — análise postural com MediaPipe

Recebe um frame e devolve os **landmarks 3D** de pontos-chave (nariz, ombros,
quadris, joelhos, tornozelos) usados para identificar movimentos fora do
padrão clínico.

**Entrada:** `frame` (imagem BGR).  
**Saída:** `{'status': 'sucesso', 'keypoints': {...}}` ou
`{'status': 'nenhum_corpo_detectado'}`.

**Lógica:** converte o frame para RGB (MediaPipe exige RGB), processa com
`pose.process()`, e, se houver landmarks, mapeia cada ponto-chave para
suas coordenadas (x, y, z). Retorna SEMPRE um dicionário.

In [ ]:
def detectar_postura(frame):
    if _mp_ok and pose is not None:
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            keypoints = {}
            nomes = ['nariz', 'ldeltoide', 'rdeltoide', 'lquadril', 'rquadril',
                     'ljoelho', 'rjoelho', 'ltobillo', 'rtobillo']
            for name, lm in zip(nomes, landmarks):
                keypoints[name] = (lm.x, lm.y, lm.z)
            return {'status': 'sucesso', 'keypoints': keypoints}
    return {'status': 'nenhum_corpo_detectado'}

## `processar_video_clinico(caminho_video, max_frames=50)` — processamento de vídeo clínico

Percorre o vídeo frame a frame combinando as duas análises de vídeo:
YOLOv8 (objetos críticos) + MediaPipe (postura). Para cada frame gera um
registro com score de anomalia, alertas e os keypoints de postura (campo
`kpts`), que alimentam a análise de movimentação depois.

**Entrada:** `caminho_video` (caminho do arquivo `.mp4`), `max_frames`
(limite de frames para a demo).  
**Saída:** lista de dicionários, um por frame, com `frame`, `objetos`,
`postura_ok`, `score_anomalia`, `alertas` e `kpts`.

**Lógica:** abre o vídeo com `cv2.VideoCapture`, e para cada frame:
- chama `detectar_objetos_yolo(frame)` (objetos/áreas críticas);
- chama `detectar_postura(frame)` (landmarks);
- avalia regras simples de postura inadequada (joelho muito esticado ou
  dobrado);
- acumula objetos críticos (confiança > 0.5);
- armazena o resultado do frame.

In [ ]:
def processar_video_clinico(caminho_video, max_frames=50):
    cap = cv2.VideoCapture(caminho_video)
    resultados = []
    frame_count = 0
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        objetos = detectar_objetos_yolo(frame)
        postura = detectar_postura(frame)
        score_anomalia = 0
        alertas = []
        kpts = None
        if postura['status'] == 'sucesso':
            kpts = postura['keypoints']
            lje = kpts.get('ljoelho', (0, 0, 0))
            rje = kpts.get('rjoelho', (0, 0, 0))
            if lje[2] > 0.8 or rje[2] > 0.8:
                score_anomalia += 1
                alertas.append('Joelho esticada')
            if lje[2] < 0.2 or rje[2] < 0.2:
                score_anomalia += 1
                alertas.append('Joelho dobrado excessivamente')
        if len(objetos) > 0:
            labels_criticos = [o['label'] for o in objetos if o['confidence'] > 0.5]
            alertas.append('Objetos detectados: %s' % ', '.join(labels_criticos[:3]))
            score_anomalia += len(labels_criticos)
        resultados.append({'frame': frame_count, 'objetos': len(objetos),
                           'postura_ok': postura['status'] == 'sucesso',
                           'score_anomalia': score_anomalia,
                           'alertas': '; '.join(alertas) if alertas else 'Nenhum',
                           'kpts': kpts})
        frame_count += 1
    cap.release()
    return resultados

## 2. Analise de Audio Medico (5 min)

## Configuração do Azure Cognitive Services

Carrega as chaves do ambiente/`.env` (via `load_dotenv`) e inicializa
Speech to Text e Text Analytics. `_azure_on` fica `True` apenas se houver
chaves, pacotes e `MODO=azure`. Imprime um diagnóstico explícito do que
está habilitado e por que (se aplicável) o Azure está desligado.

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

try:
    import azure.cognitiveservices.speech as speechsdk
    from azure.ai.textanalytics import TextAnalyticsClient
    from azure.core.credentials import AzureKeyCredential
    _HAS_AZURE_LIBS = True
except Exception:
    _HAS_AZURE_LIBS = False

speech_key = os.getenv("AZURE_SPEECH_KEY", "")
speech_region = os.getenv("AZURE_SPEECH_REGION", "westus2")
text_key = os.getenv("AZURE_TEXT_ANALYTICS_KEY", "")
text_endpoint = os.getenv("AZURE_TEXT_ANALYTICS_ENDPOINT", "").strip()

_endpoint_efetivo = text_endpoint or (
    "https://%s.api.cognitive.microsoft.com/" % text_region)

_azure_on = bool(AZURE_ON and _HAS_AZURE_LIBS and speech_key and text_key)
if _azure_on:
    speech_config = speechsdk.SpeechConfig(subscription=speech_key,
                                           region=speech_region)
    speech_config.speech_recognition_language = "pt-BR"
    text_analytics_client = TextAnalyticsClient(
        endpoint=_endpoint_efetivo,
        credential=AzureKeyCredential(text_key))
else:
    speech_config = None
    text_analytics_client = None

print("--- Condicao do Azure ---")
print("  pacotes Azure instalados :", _HAS_AZURE_LIBS)
print("  MODO == 'azure'          :", AZURE_ON)
print("  AZURE_SPEECH_KEY definida:", bool(speech_key))
print("  AZURE_TEXT_ANALYTICS_KEY :", bool(text_key))
print("  Endpoint Text Analytics  :", _endpoint_efetivo,
      "(do .env)" if text_endpoint else "(legado por regiao)")
if _azure_on:
    print("=> Azure LIGADO (Speech to Text + Text Analytics)")
else:
    faltas = []
    if not AZURE_ON:
        faltas.append("MODO=offline (nao pediu azure)")
    if not _HAS_AZURE_LIBS:
        faltas.append("pacotes Azure ausentes")
    if not speech_key:
        faltas.append("falta AZURE_SPEECH_KEY")
    if not text_key:
        faltas.append("falta AZURE_TEXT_ANALYTICS_KEY")
    print("=> Azure DESLIGADO, usando OFFLINE. Motivos:",
          "; ".join(faltas) or "-")

## `transcrever_audio(caminho_audio)` — Speech to Text (Azure)

Converte um arquivo de áudio em texto usando o **Azure Speech to Text**.

**Entrada:** `caminho_audio` (caminho do arquivo `.wav`).  
**Saída:** string com o texto transcrito, ou `""` se não houver áudio/
chave Azure.

**Lógica:** se `_azure_on` estiver ativo, configura um
`SpeechRecognizer` sobre o arquivo e chama `recognize_once()`, retornando
o texto reconhecido. Se não estiver no modo Azure, retorna string vazia e
registra que foi pulado. Cada resultado é gravado no `REGISTRO_API`.

In [ ]:
def transcrever_audio(caminho_audio):
    if not _azure_on:
        registrar_api("Azure Speech",
                      {"status": "skipped", "motivo": "_azure_on=False",
                       "texto": ""})
        return ""
    try:
        audio_config = speechsdk.AudioConfig(filename=caminho_audio)
        speech_recognizer = speechsdk.SpeechRecognizer(
            speech_config=speech_config, audio_config=audio_config)
        result = speech_recognizer.recognize_once()
        motivo = str(result.reason)
        if result.reason == speechsdk.ResultReason.RecognizedSpeech:
            registrar_api("Azure Speech",
                          {"status": "ok", "arquivo": caminho_audio,
                           "reason": motivo, "texto": result.text})
            print(f"Transcricao do audio ({caminho_audio}): {result.text}")
            return result.text
        registrar_api("Azure Speech",
                      {"status": "sem_fala", "arquivo": caminho_audio,
                       "reason": motivo})
        return ""
    except Exception as e:
        registrar_api("Azure Speech",
                      {"status": "erro", "arquivo": caminho_audio,
                       "erro": str(e)})
        print(f"Erro na transcricao: {e}")
        return ""

## `analisar_texto_azure(texto)` — sentimento e entidades (Azure Text Analytics)

Analisa o texto transcrito extraindo sentimento (positivo/neutro/negativo)
e entidades médicas relevantes (Pessoa, Organização, Quantidade,
Temperatura, DateTime).

**Entrada:** `texto` (string transcrita).  
**Saída:** dicionário com `sentimento` (`rotulo` + `pontuacoes`) e
`entidades` (lista).

**Lógica:** se não houver Azure ativo, usa uma heurística simples baseada
em palavras-chave clínicas, ignorando termos negados (ex.: "nega dor").
Se houver Azure, chama `analyze_sentiment` e `recognize_entities` e
filtra apenas entidades de interesse clínico. Todo retorno é registrado
no log.

In [ ]:
def analisar_texto_azure(texto):
    padrao = {'sentimento': {'rotulo': 'neutro',
                             'pontuacoes': {'positivo': 0.0,
                                            'negativo': 0.0,
                                            'neutro': 1.0}},
              'entidades': []}
    if not texto or len(texto.strip()) < 3:
        registrar_api("Text Analytics",
                      {"status": "skipped", "motivo": "texto vazio/curto"})
        return padrao
    if not _azure_on:
        import re as _re
        t = texto.lower()
        neg_tokens = ['nega', 'negou', 'negando', 'nao', 'não', 'sem']
        clausulas = [cl for cl in _re.split(r'[.;,]', t) if cl.strip()]
        negativos = ['dor', 'cansado', 'fadiga', 'falta de ar', 'exausto',
                     'complicacao', 'desconforto', 'socorro']
        _casa = lambda cl, w: _re.search(r'\b' + _re.escape(w) + r'\b', cl)
        neg = any(_casa(cl, w) and not any(nt in cl for nt in neg_tokens)
                  for cl in clausulas for w in negativos)
        resultado = {'sentimento': {'rotulo': ('negativo' if neg else 'neutro'),
                                    'pontuacoes': {'positivo': 0.0,
                                                   'negativo': (1.0 if neg else 0.0),
                                                   'neutro': (0.0 if neg else 1.0)}},
                     'entidades': []}
        registrar_api("Text Analytics (offline)",
                      {"status": "heuristica", **resultado})
        print(f"Analise de texto (offline): {resultado['sentimento']['rotulo']}, "
              f"entidades: {resultado['entidades']}")
        return resultado
    try:
        sentiment_result = text_analytics_client.analyze_sentiment([texto])[0]
        entity_result = text_analytics_client.recognize_entities([texto])[0]
    except Exception as e:
        msg = str(e)
        registrar_api("Text Analytics (Azure)", {"status": "erro", "erro": msg})
        print(f"Azure Text Analytics indisponivel: {msg}")
        if "unauthorized" in msg.lower() or "401" in msg:
            print("  -> 401: chave invalida OU endpoint legado num recurso novo. "
                  "Preencha AZURE_TEXT_ANALYTICS_ENDPOINT com o endpoint do recurso.")
        if "forbidden" in msg.lower() or "403" in msg:
            print("  -> 403 Forbidden: endpoint de outro produto "
                  "(ex.: ...search.windows.net = Azure AI SEARCH) ou firewall "
                  "do recurso bloqueando.")
        return padrao
    entidades_medicas = []
    for entity in entity_result.entities:
        if entity.category in ['Person', 'Organization', 'Quantity',
                               'Temperature', 'DateTime']:
            entidades_medicas.append({'texto': entity.text,
                                      'tipo': entity.category,
                                      'confianca': entity.confidence_score})
    cs = sentiment_result.confidence_scores
    resultado = {'sentimento': {'rotulo': sentiment_result.sentiment,
                                'pontuacoes': {'positivo': float(cs.positive),
                                               'negativo': float(cs.negative),
                                               'neutro': float(cs.neutral)}},
                 'entidades': entidades_medicas}
    registrar_api("Text Analytics (Azure)", {"status": "ok", **resultado})
    return resultado

## `detectar_alteracoes_vocais(texto_transcrito)` — alterações vocais

Procura no texto transcrito sinais de condições médicas (fadiga, dificuldade
respiratória, dor). Usa **cláusulas** (divisão por pontuação) e **ignora
menções negadas** (ex.: "nega dor", "sem falta de ar"), evitando
falso-positivo quando o paciente está sem sintomas. Usa **fronteira de
palavra** (``) para não casar palavras dentro de outras
(ex.: `dor` não casa dentro de `dormiu`).

**Entrada:** `texto_transcrito` (string).  
**Saída:** dicionário com uma chave por condição (`fadiga`,
`dificuldade_respiratoria`, `dor`), cada uma com `presente`, `encontradas`
e `quantidade`. 

In [ ]:
def detectar_alteracoes_vocais(texto_transcrito):
    import re as _re
    padroes_clinicos = {
        'fadiga': ['cansado', 'fadiga', 'sono', 'exausto'],
        'dificuldade_respiratoria': ['dificuldade para respirar', 'falta de ar'],
        'dor': ['dor', 'complicacao'],
    }
    neg_tokens = ['nega', 'negou', 'negando', 'nao', 'não', 'sem', 'sem sinais']
    texto_lower = (texto_transcrito or '').lower()
    clausulas = [c for c in _re.split(r'[.;,]', texto_lower) if c.strip()]
    resultados = {}
    for condicao, palavras in padroes_clinicos.items():
        encontradas = []
        for cl in clausulas:
            if any(nt in cl for nt in neg_tokens):
                continue
            for p in palavras:
                if _re.search(r'\b' + _re.escape(p) + r'\b', cl)                    and p not in encontradas:
                    encontradas.append(p)
        resultados[condicao] = {'presente': len(encontradas) > 0,
                                'encontradas': encontradas,
                                'quantidade': len(encontradas)}
    return resultados

## `detectar_anomalias_sinais_vitais(df_sinais, janela=6)` — sinais vitais

Detecta anomalias em séries temporais de sinais vitais usando **faixas
clínicas no último valor** + estatística da janela recente (máx/mín/média).
Colunas suportadas: `fc`, `spo2`, `pa_sist`, `temp` (usa as que existirem).

**Entrada:** `df_sinais` (DataFrame pandas).  
**Saída:** dicionário com flags `fc_anomalia`, `bp_anomalia`,
`spo2_anomalia`, `temp_anomalia`, `alertas` (apenas anomalias reais),
`resumo` (uma linha por parâmetro, incluindo o estado estável) e
`paciente_estavel`.

**Faixas clínicas (último valor):**
- FC: 60–100 bpm (alerta <50 ou >110)
- SpO2: ≥95% (alerta <95)
- PA sistólica: 90–140 mmHg (alerta <90 ou >140)
- Temperatura: 35,5–38,0 °C (alerta >38 ou <35,5)

In [ ]:
def detectar_anomalias_sinais_vitais(df_sinais, janela=6):
    anomalia = {'fc_anomalia': False, 'bp_anomalia': False,
                'spo2_anomalia': False, 'temp_anomalia': False,
                'alertas': [], 'resumo': []}

    def _serie(col):
        if col not in df_sinais.columns:
            return None
        s = pd.to_numeric(df_sinais[col], errors='coerce').dropna()
        return None if s.empty else s

    def _estat(s):
        w = s.iloc[-janela:]
        return {'ultimo': float(s.iloc[-1]), 'media': float(w.mean()),
                'min': float(w.min()), 'max': float(w.max())}

    # FC (faixa 60-100; alerta <50 ou >110)
    fc = _serie('fc')
    if fc is not None:
        e = _estat(fc)
        grave = e['ultimo'] < 50 or e['ultimo'] > 110
        anomalia['fc_anomalia'] = grave
        anomalia['resumo'].append("FC: ultimo %d bpm (janela %d-%d, media %d)"
            % (round(e['ultimo']), round(e['min']), round(e['max']),
               round(e['media'])))
        if grave:
            anomalia['alertas'].append("FC fora da faixa: %d bpm"
                                       % round(e['ultimo']))
    # SpO2 (>=95)
    spo2 = _serie('spo2')
    if spo2 is not None:
        e = _estat(spo2); grave = e['ultimo'] < 95
        anomalia['spo2_anomalia'] = grave
        anomalia['resumo'].append("SpO2: ultimo %d%% (janela %d-%d, media %d)"
            % (round(e['ultimo']), round(e['min']), round(e['max']),
               round(e['media'])))
        if grave:
            anomalia['alertas'].append("SpO2 baixa: %d%%" % round(e['ultimo']))
    # PA sistolica (90-140)
    pa = _serie('pa_sist')
    if pa is not None:
        e = _estat(pa); grave = e['ultimo'] < 90 or e['ultimo'] > 140
        anomalia['bp_anomalia'] = grave
        anomalia['resumo'].append("PA sistolica: ultimo %d mmHg "
            "(janela %d-%d, media %d)" % (round(e['ultimo']),
            round(e['min']), round(e['max']), round(e['media'])))
        if grave:
            anomalia['alertas'].append("PA fora da faixa: %d mmHg"
                                       % round(e['ultimo']))
    # Temperatura (35.5-38.0)
    temp = _serie('temp')
    if temp is not None:
        e = _estat(temp); febre = e['ultimo'] > 38.0; hipo = e['ultimo'] < 35.5
        anomalia['temp_anomalia'] = bool(febre or hipo)
        rot = "Febre" if febre else ("Hipotermia" if hipo else
                                     "Temperatura normal")
        anomalia['resumo'].append("Temperatura: ultimo %.1f C - %s "
            "(janela %.1f-%.1f)" % (e['ultimo'], rot, e['min'], e['max']))
        if febre or hipo:
            anomalia['alertas'].append("Temperatura alterada: %.1f C"
                                       % e['ultimo'])

    base = fc if fc is not None else (spo2 if spo2 is not None else None)
    anomalia['n_leituras'] = int(len(base)) if base is not None else 0
    anomalia['parametros'] = [n for n in ('fc', 'spo2', 'pa_sist', 'temp')
                              if _serie(n) is not None]
    anomalia['paciente_estavel'] = not any([anomalia['fc_anomalia'],
        anomalia['bp_anomalia'], anomalia['spo2_anomalia'],
        anomalia['temp_anomalia']])
    if anomalia['paciente_estavel']:
        anomalia['resumo'].append("Sinais vitais estaveis: paciente sem "
            "sintomas agudos")
    return anomalia

## `detectar_anomalias_prescricoes(lista_prescricoes)` — prescrições

Analisa a evolução de prescrições comparando a **última com a anterior**:
novo medicamento, medicamento suspenso, dosagem alterada e possível
interação (novo + dosagem alterada). Também resume a linha do tempo
(medicamentos iniciados/suspensos no período e prescrição atual).

**Entrada:** `lista_prescricoes` (lista de dicionários
`{medicamento: dosagem}`).  
**Saída:** dicionário com flags `medicamento_novo`, `dosagem_alterada`,
`interacao_drogas`, `estavel`, `alertas` (só alterações reais) e `resumo`
(linha do tempo + estado).

In [ ]:
def detectar_anomalias_prescricoes(lista_prescricoes):
    anomalia = {'medicamento_novo': False, 'dosagem_alterada': False,
                'interacao_drogas': False, 'estavel': False,
                'alertas': [], 'resumo': []}
    if not isinstance(lista_prescricoes, list) or len(lista_prescricoes) < 2:
        anomalia['alertas'].append('Historico insuficiente')
        return anomalia
    atual, anterior = lista_prescricoes[-1], lista_prescricoes[-2]
    ma = {str(k).lower(): (k, v) for k, v in atual.items()}
    mb = {str(k).lower(): (k, v) for k, v in anterior.items()}
    novos = set(ma) - set(mb)
    retirados = set(mb) - set(ma)
    alter = [('%s (%s -> %s)' % (ma[k][0], mb[k][1], ma[k][1]))
             for k in set(ma) & set(mb) if str(ma[k][1]) != str(mb[k][1])]
    if novos:
        anomalia['medicamento_novo'] = True
        anomalia['alertas'].append('Novo medicamento: ' + ', '.join(
            ma[n][0] for n in sorted(novos)))
    for r in sorted(retirados):
        anomalia['alertas'].append('Medicamento suspenso: ' + mb[r][0])
    if alter:
        anomalia['dosagem_alterada'] = True
        for a in sorted(alter):
            anomalia['alertas'].append('Dosagem alterada: ' + a)
    if anomalia['medicamento_novo'] and anomalia['dosagem_alterada']:
        anomalia['interacao_drogas'] = True
        anomalia['alertas'].append('WARNING: possivel interacao (novo '
                                   'medicamento + dosagem alterada)')
    if not novos and not retirados and not alter:
        anomalia['estavel'] = True
    inicio = lista_prescricoes[0]
    ini_keys = {str(k).lower() for k in inicio}
    fim_keys = set(ma)
    iniciados = sorted({ma[k][0] for k in fim_keys - ini_keys})
    suspensos = sorted({str(k) for k in inicio
                        if str(k).lower() not in fim_keys})
    ativos = ', '.join('%s (%s)' % (k, v) for k, v in atual.items())
    anomalia['resumo'].append('Evolucao de prescricoes (%d etapas): '
        'iniciados=%s | suspensos=%s' %
        (len(lista_prescricoes), ', '.join(iniciados) or '-',
         ', '.join(suspensos) or '-'))
    anomalia['resumo'].append('Prescricao atual: ' + (ativos or '-'))
    if anomalia['estavel']:
        anomalia['resumo'].append('Ultima evolucao estavel: sem alteracoes')
    return anomalia

## `detectar_anomalias_movimentacao(keypoints_history)` — movimento

Analisa o padrão de movimentação a partir dos keypoints de postura
(frame a frame). Mede o deslocamento do centroide dos quadris e sinaliza:
movimento repentino, imobilidade anormal e padrão suspeito de queda
(movimento repentino seguido de imobilidade).

**Entrada:** `keypoints_history` (lista de dicionários com `lquadril`/
`rquadril`).  
**Saída:** dicionário com flags `movimento_repetino`,
`imobilidade_anormal`, `quedas_suspeitas` e `alertas`.

In [ ]:
def detectar_anomalias_movimentacao(keypoints_history):
    anomalia = {'movimento_repetino': False, 'imobilidade_anormal': False,
                'quedas_suspeitas': False, 'alertas': []}
    if len(keypoints_history) < 2:
        anomalia['alertas'].append('Historico insuficiente')
        return anomalia
    centoidos = []
    for kp_data in keypoints_history:
        lq = kp_data.get('lquadril', (0, 0))
        rq = kp_data.get('rquadril', (0, 0))
        centroide = ((lq[0] + rq[0]) / 2, (lq[1] + rq[1]) / 2)
        centoidos.append(centroide)
    if len(centoidos) >= 2:
        velocidades = []
        for i in range(1, len(centoidos)):
            dist = ((centoidos[i][0] - centoidos[i-1][0])**2
                    + (centoidos[i][1] - centoidos[i-1][1])**2) ** 0.5
            velocidades.append(dist)
        if velocidades:
            velocidade_max = max(velocidades)
            velocidade_media = sum(velocidades) / len(velocidades)
            if velocidade_max > 3 * velocidade_media and velocidade_media > 0:
                anomalia['movimento_repetino'] = True
                anomalia['alertas'].append('Movimento repentino: dist %.3f'
                                           % velocidade_max)
            N = min(10, len(velocidades))
            ultimas = velocidades[-N:]
            vm = sum(ultimas) / len(ultimas) if ultimas else 0
            if vm < 0.05 and len(ultimas) >= 10:
                anomalia['imobilidade_anormal'] = True
                anomalia['alertas'].append('Imobilidade: %d frames '
                                           'sem movimento'
                                           % len([v for v in ultimas
                                                  if v < 0.01]))
            if anomalia['movimento_repetino'] and                anomalia['imobilidade_anormal']:
                anomalia['quedas_suspeitas'] = True
                anomalia['alertas'].append('WARNING: Padrao de queda '
                                           'suspeito: movimento repentino '
                                           '+ imobilidade')
    return anomalia

## `workflow_integrado(resultado_video, resultado_audio, resultado_sinais)` — decisão final

Integra vídeo + áudio + anomalias em uma **única decisão clínica**. Cada
fonte contribui com pontos de risco: vídeo = 2 por alerta, áudio = 3 por
alteração vocal, sinais = 3 por anomalia. A soma é convertida em nível:
Normal (<3), ALTO (3-5) ou CRÍTICO (≥6), com ação recomendada.

**Entrada:** os três dicionários de resultado.  
**Saída:** dicionário com `decisao`, `nivel`, `acao_recomendada`,
`total_alertas`, `alertas` e `punteis_risco`. Soma risco **só quando há
alertas reais** (não conta apenas “estável”).

In [ ]:
def workflow_integrado(resultado_video,
                                                        resultado_audio,
                                                        resultado_sinais):
    alertas_gerais = []
    punteis_risco = 0
    if resultado_video and 'total_alertas' in resultado_video:
        vs = resultado_video.get('alertas_listas', [])
        sig = [a for a in vs if 'Nenhum' not in a]
        if sig:
            alertas_gerais.extend(sig)
            punteis_risco += len(sig) * 2
    if resultado_audio and 'total_alteracoes' in resultado_audio:
        aps = [k for k, v in resultado_audio['alteracoes'].items()
               if v.get('presente', False)]
        if aps:
            mapa = {'fadiga': 'Paciente relata fadiga',
                    'dificuldade_respiratoria':
                        'Paciente relata dificuldade respiratoria',
                    'dor': 'Paciente relata dor',
                    'alerta_urgencia':
                        'ALERTA URGENCIA: solicitacao de ajuda'}
            for a in aps:
                m = mapa.get(a, a)
                alertas_gerais.append(m)
                punteis_risco += 3
    if resultado_sinais and resultado_sinais.get('alertas'):
        alertas_gerais.extend(resultado_sinais['alertas'])
        punteis_risco += 3
    if punteis_risco >= 6:
        decisao = 'ALERTA MÉDICO IMEDIATO'
        nivel = 'CRITICO'
        acao = 'Contatar equipe médica imediatamente'
    elif punteis_risco >= 3:
        decisao = 'ALERTA - Revisao necessaria'
        nivel = 'ALTO'
        acao = 'Avaliar paciente e monitorar 30 min'
    else:
        decisao = 'Monitoramento normal continuo'
        nivel = 'Normal'
        acao = 'Continuar rotina de cuidados'
    return {'decisao': decisao, 'nivel': nivel,
            'acao_recomendada': acao,
            'total_alertas': len(alertas_gerais),
            'alertas': '; '.join(alertas_gerais)
            if alertas_gerais else 'Nenhum motivo de alerta',
            'punteis_risco': punteis_risco}

## 3. Captura / seleção da fonte dos arquivos
Na execução, cada mídia (vídeo e áudio) pode: **(1)** ser **capturada** ao vivo do
dispositivo (parar ao teclar), **salvando** em `capturas/` e seguindo para análise;
**(2)** receber um **caminho** de arquivo existente; ou **(3)** usar o **mock simulado**
(offline). A escolha define `CAMINHO_VIDEO`/`CAMINHO_AUDIO` que o pipeline consome a seguir.
No modo offline o áudio capturado é transcrito de verdade com **Vosk (pt-BR)**; no modo
Azure usa-se o **Speech to Text**.

## Captura / seleção da fonte dos arquivos

Cada mídia (vídeo e áudio) pode: **(1) capturar** ao vivo do dispositivo
(parar ao teclar), salvando em `capturas/`; **(2) receber um caminho**
de arquivo existente; ou **(3) usar o mock simulado** (offline).
A escolha define `CAMINHO_VIDEO`/`CAMINHO_AUDIO`. No modo offline o
áudio capturado é transcrito com Vosk (pt-BR); no modo Azure usa
Speech to Text.

In [ ]:
import wave, datetime

PASTA_CAPTURAS = "capturas"
os.makedirs(PASTA_CAPTURAS, exist_ok=True)

def capturar_video(fonte=0, fps=20):
    cap = cv2.VideoCapture(fonte)
    if not cap.isOpened():
        raise RuntimeError("camera indisponivel")
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 640
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 480
    saida = os.path.join(PASTA_CAPTURAS,
                         "video_%s.mp4"
                         % datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
    writer = cv2.VideoWriter(saida,
                             cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    print("Gravando VIDEO... pressione 'q' na janela para PARAR.")
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            writer.write(frame)
            cv2.imshow("Captura de video (q para parar)", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release(); writer.release(); cv2.destroyAllWindows()
    return saida

def capturar_audio(sr=16000, canais=1):
    import sounddevice as sd
    saida = os.path.join(PASTA_CAPTURAS,
                         "audio_%s.wav"
                         % datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
    frames = []
    def _cb(ind, out, t, status):
        frames.append(ind.copy())
    print("Gravando AUDIO... pressione ENTER para PARAR.")
    with sd.InputStream(samplerate=sr, channels=canais,
                        dtype="int16", callback=_cb):
        input()
    dados = np.concatenate(frames) if frames         else np.zeros((0, canais), np.int16)
    with wave.open(saida, "wb") as wf:
        wf.setnchannels(canais); wf.setsampwidth(2); wf.setframerate(sr)
        wf.writeframes(dados.tobytes())
    return saida

VOSK_DIR = os.path.join("modelos", "vosk-pt")
def _baixar_vosk(url="https://alphacephei.com/vosk/models/"
                   "vosk-model-small-pt-0.3.zip"):
    import zipfile
    os.makedirs("modelos", exist_ok=True)
    zipbuf = os.path.join("modelos", "vosk-pt.zip")
    baixou = False
    try:
        print("Baixando modelo Vosk pt-BR (uma vez)...")
        try:
            import requests, certifi
            with requests.get(url, stream=True,
                              verify=certifi.where(), timeout=180) as r:
                r.raise_for_status()
                with open(zipbuf, "wb") as f:
                    for pedaco in r.iter_content(chunk_size=1 << 20):
                        if pedaco:
                            f.write(pedaco)
            baixou = True
        except ImportError:
            import urllib.request, ssl, certifi
            ctx = ssl.create_default_context(cafile=certifi.where())
            with urllib.request.urlopen(url, context=ctx) as resp,                     open(zipbuf, "wb") as f:
                f.write(resp.read())
            baixou = True
    except Exception as e:
        print("Falha ao baixar modelo Vosk:", e)
    if baixou:
        try:
            with zipfile.ZipFile(zipbuf) as z:
                raiz = z.namelist()[0].split("/")[0]
                z.extractall("modelos")
            destino = os.path.join("modelos", raiz)
            if not os.path.isdir(VOSK_DIR):
                os.replace(destino, VOSK_DIR)
            print("Modelo Vosk pronto em", VOSK_DIR)
        except Exception as e:
            print("Falha ao extrair o modelo Vosk:", e)
    if not os.path.isdir(VOSK_DIR):
        print("Alternativa: baixe manualmente", url)
        print("  e extraia o conteudo para a pasta:", VOSK_DIR)

def transcrever_audio_offline(caminho_audio):
    try:
        import json as _json
        from vosk import Model, KaldiRecognizer
    except Exception as e:
        registrar_api("Vosk (offline)",
                      {"status": "indisponivel", "erro": str(e)})
        print("Vosk nao instalado -> sem transcricao offline "
              "(pip install vosk).")
        return ""
    if not os.path.isdir(VOSK_DIR):
        _baixar_vosk()
    if not os.path.isdir(VOSK_DIR):
        return ""
    model = Model(VOSK_DIR)
    rec = KaldiRecognizer(model, 16000)
    pedacos = []
    with wave.open(caminho_audio, "rb") as wf:
        if wf.getnchannels() != 1 or wf.getsampwidth() != 2:
            print("Vosk espera WAV 16kHz mono PCM; arquivo pode nao "
                  "transcrever bem.")
        while True:
            dados = wf.readframes(4000)
            if not dados:
                break
            if rec.AcceptWaveform(dados):
                pedacos.append(_json.loads(rec.Result()).get("text", ""))
    pedacos.append(_json.loads(rec.FinalResult()).get("text", ""))
    texto_final = " ".join(p for p in pedacos if p).strip()
    registrar_api("Vosk (offline)",
                  {"status": "ok", "arquivo": caminho_audio,
                   "segmentos": len(pedacos), "texto": texto_final})
    return texto_final

def _escolher_fonte(nome):
    try:
        r = input("[%s] 1=captura  2=inserir caminho  3=simulado "
                  "(Enter=3): " % nome).strip()
    except Exception:
        r = ""
    return r or "3"

def resolver_fonte_video():
    global CAMINHO_VIDEO
    e = _escolher_fonte("VIDEO")
    CAMINHO_VIDEO = ""
    if e == "1":
        try:
            CAMINHO_VIDEO = capturar_video()
            print("Video capturado em:", CAMINHO_VIDEO)
        except Exception as err:
            print("Falha na captura de video -> simulado.", err)
            CAMINHO_VIDEO = ""
    elif e == "2":
        p = input("  caminho do arquivo de video (.mp4): ").strip()
        CAMINHO_VIDEO = p if p and os.path.exists(p) else ""
        if not CAMINHO_VIDEO:
            print("Arquivo nao encontrado -> simulado.")

def resolver_fonte_audio():
    global CAMINHO_AUDIO, TEXTO_AUDIO_CAPTURADO
    TEXTO_AUDIO_CAPTURADO = ""
    e = _escolher_fonte("AUDIO")
    CAMINHO_AUDIO = ""
    if e == "1":
        try:
            CAMINHO_AUDIO = capturar_audio()
            print("Audio capturado em:", CAMINHO_AUDIO)
        except Exception as err:
            print("Falha na captura de audio -> simulado.", err)
            CAMINHO_AUDIO = ""
    elif e == "2":
        p = input("  caminho do arquivo de audio (.wav): ").strip()
        CAMINHO_AUDIO = p if p and os.path.exists(p) else ""
        if not CAMINHO_AUDIO:
            print("Arquivo nao encontrado -> simulado.")
    if CAMINHO_AUDIO:
        t = transcrever_audio(CAMINHO_AUDIO) if _azure_on             else transcrever_audio_offline(CAMINHO_AUDIO)
        if t:
            TEXTO_AUDIO_CAPTURADO = t
            print("Transcricao:", TEXTO_AUDIO_CAPTURADO)

resolver_fonte_video()
resolver_fonte_audio()
print("Fontes -> video:", CAMINHO_VIDEO or "(simulado)",
      "| audio:", CAMINHO_AUDIO or "(simulado)")

## 4. Execução do Pipeline Multimodal (vídeo + áudio + anomalias)
Gera `resultado_video`, `resultado_audio` e `resultado_sinais` que alimentam o relatório/decisão final.

## `executar_video()` — execução da análise de vídeo

Processa o vídeo clínico (ou frames sintéticos se não houver arquivo) e
retorna `resultado_video` com `frames`, `objetos_totais`,
`anomalias_posturais`, `alertas_listas`, `total_alertas` e
`keypoints_history` (para a análise de movimentação).

**Entrada:** `CAMINHO_VIDEO` (env ou `dados/video.mp4`).  
**Saída:** dicionário de agregação do vídeo.

In [ ]:
def gerar_frames_sinteticos(n=40):
    """Frames simulados para rodar YOLO/MediaPipe sem arquivo real."""
    frames = []
    for i in range(n):
        base = np.full((480, 640, 3), 60 + i, dtype=np.uint8)
        frames.append(base)
    return frames

def executar_video():
    resultados = None
    if os.path.exists(CAMINHO_VIDEO):
        try:
            resultados = processar_video_clinico(CAMINHO_VIDEO)
        except Exception as e:
            print("Falha no video real, usando frames sinteticos:", e)
            resultados = None
    if not resultados:
        resultados = []
        for idx, frame in enumerate(gerar_frames_sinteticos()):
            objetos = detectar_objetos_yolo(frame)
            postura = detectar_postura(frame)
            alertas, score = [], 0
            kpts = None
            if postura.get('status') == 'sucesso':
                kpts = postura.get('keypoints', {})
                lje = kpts.get('ljoelho', (0, 0, 0))
                rje = kpts.get('rjoelho', (0, 0, 0))
                if lje[2] > 0.8 or rje[2] > 0.8:
                    score += 1
                    alertas.append('Joelho esticado')
                if lje[2] < 0.2 or rje[2] < 0.2:
                    score += 1
                    alertas.append('Joelho dobrado excessivamente')
            if objetos:
                labels = [o['label'] for o in objetos if o['confidence'] > 0.5]
                if labels:
                    score += len(labels)
                    alertas.append('Objetos detectados: '
                                   + ', '.join(labels[:3]))
            resultados.append({'frame': idx, 'objetos': len(objetos),
                               'postura_ok': postura.get('status')
                               == 'sucesso',
                               'score_anomalia': score,
                               'alertas': '; '.join(alertas)
                               if alertas else 'Nenhum',
                               'kpts': kpts})
    alertas_listas = [r['alertas'] for r in resultados
                      if r['alertas'] != 'Nenhum']
    keypoints_history = [r['kpts'] for r in resultados
                         if r.get('kpts')]
    return {'frames': len(resultados),
            'objetos_totais': sum(r['objetos'] for r in resultados),
            'anomalias_posturais':
                sum(r['score_anomalia'] for r in resultados),
            'alertas_listas': alertas_listas,
            'total_alertas': len(alertas_listas),
            'keypoints_history': keypoints_history}

resultado_video = executar_video()
print("Video: {} frames | {} alertas | {} frames c/ postura"
      .format(resultado_video['frames'],
              resultado_video['total_alertas'],
              len(resultado_video['keypoints_history'])))

## `executar_audio()` — execução da análise de áudio

Usa a transcrição já capturada na etapa de fonte (Azure ou Vosk); se houver
arquivo de áudio e ainda não transcreveu, transcreve agora. Sem arquivo,
usa `dados/consulta.txt` ou uma frase padrão sem sintomas. Cada passo
é registrado no log.

**Entrada:** `CAMINHO_AUDIO`, `dados/consulta.txt` ou frase padrão.  
**Saída:** dicionário com `transcricao`, `sentimento`, `entidades`,
`alteracoes` e `total_alteracoes`.

In [ ]:
def executar_audio():
    texto = globals().get("TEXTO_AUDIO_CAPTURADO", "")
    origem = "captura/transcricao"
    if not texto and CAMINHO_AUDIO and os.path.exists(CAMINHO_AUDIO):
        texto = transcrever_audio(CAMINHO_AUDIO) if _azure_on             else transcrever_audio_offline(CAMINHO_AUDIO)
    if not texto:
        arq_txt = os.path.join("dados", "consulta.txt")
        if os.path.exists(arq_txt):
            with open(arq_txt, encoding="utf-8") as f:
                texto = f.read().strip()
            origem = arq_txt
    if not texto:
        texto = ("Paciente: nego dor e falta de ar; sinto-me bem e sem "
                 "sintomas hoje.")
        origem = "frase padrao (sem sintomas)"
    registrar_api("Audio (fonte)",
                  {"modo": MODO, "origem": origem, "texto": texto})
    analise = analisar_texto_azure(texto)
    alteracoes = detectar_alteracoes_vocais(texto)
    return {'transcricao': texto,
            'sentimento': analise.get('sentimento', {}),
            'entidades': analise.get('entidades', []),
            'alteracoes': alteracoes,
            'total_alteracoes':
                sum(1 for v in alteracoes.values()
                    if v.get('presente'))}

resultado_audio = executar_audio()
print("Audio: sentimento=%s | %d alteracoes vocais"
      % (resultado_audio['sentimento'].get('rotulo', 'neutro'),
         resultado_audio['total_alteracoes']))

## Carregadores de dados e sintético de movimento

Carrega os arquivos de entrada do paciente (`dados/sinais.csv` e
`dados/prescricoes.json`), com fallback para exemplo se não existirem.
Também gera **keypoints sintéticos variados** quando não há vídeo real,
com cenário configurável via `MOVIMENTO_CENARIO` (`estavel` padrão ou
`queda`) e `SEMENTE` para determinismo.

In [ ]:
import json as _json

ARQ_SINAIS = os.getenv("CAMINHO_SINAIS", os.path.join("dados", "sinais.csv"))
ARQ_PRESCRICOES = os.getenv("CAMINHO_PRESCRICOES",
                            os.path.join("dados", "prescricoes.json"))

def carregar_sinais_vitais(caminho=None):
    caminho = caminho or ARQ_SINAIS
    if caminho and os.path.exists(caminho):
        try:
            df = pd.read_csv(caminho)
            if {"fc", "spo2"}.issubset(df.columns) and len(df) >= 1:
                registrar_api("load:sinais_vitais",
                              {"caminho": caminho, "linhas": int(len(df)),
                               "colunas": list(df.columns)})
                return df, "dataset(%s)" % caminho
        except Exception as e:
            print("Falha ao ler sinais.csv:", e)
    registrar_api("load:sinais_vitais",
                  {"caminho": "%s -> EXEMPLO" % (caminho or "-")})
    df = pd.DataFrame({"fc": [72, 75, 74, 78, 80, 79, 76],
                       "spo2": [97, 98, 97, 98, 97, 98, 98]})
    return df, "EXEMPLO"

def carregar_prescricoes(caminho=None):
    caminho = caminho or ARQ_PRESCRICOES
    if caminho and os.path.exists(caminho):
        try:
            with open(caminho, encoding="utf-8") as f:
                dados = _json.load(f)
            if isinstance(dados, list) and dados:
                registrar_api("load:prescricoes",
                              {"caminho": caminho, "etapas": len(dados)})
                return dados, "dataset(%s)" % caminho
        except Exception as e:
            print("Falha ao ler prescricoes.json:", e)
    registrar_api("load:prescricoes",
                  {"caminho": "%s -> EXEMPLO" % (caminho or "-")})
    return ([{"Dipirona": "500mg"}, {"Dipirona": "500mg"}], "EXEMPLO")

def gerar_keypoints_sinteticos(n=9, semente=None, cenario="estavel"):
    rng = np.random.default_rng(semente)
    hist, x, y = [], 0.5, 0.5
    for i in range(n):
        if cenario == "queda" and i == n // 2:
            x = float(np.clip(x + 0.6, 0, 1))
            y = float(np.clip(y + 0.5, 0, 1))
        elif cenario == "queda" and i > n // 2:
            pass
        else:
            x = float(np.clip(x + rng.normal(0, 0.06), 0, 1))
            y = float(np.clip(y + rng.normal(0, 0.06), 0, 1))
        hist.append({"lquadril": (x, y), "rquadril": (x + 0.05, y)})
    return hist

## `executar_sinais()` — detecção de anomalias completa

Alimenta os três detectores (sinais vitais, prescrições, movimentação) e
consolida tudo em `resultado_sinais` com:
- `alertas`: apenas anomalias reais (usado pelo `workflow_integrado`);
- `resumo`: linhas descritivas de cada parâmetro (inclui "estável");
- `fontes`: de onde veio cada dado (dataset/EXEMPLO/VIDEO/SINTETICO);
- `paciente_estavel`: True se não houver anomalia nos sinais nem
  movimento/queda.

**Lógica:** carrega sinais e prescrições dos arquivos (ou exemplo);
usa os keypoints do vídeo para movimentação ou trajetória sintética
variada se não houver vídeo.

In [ ]:
def executar_sinais():
    fontes = {}
    df_sinais, fontes["sinais_vitais"] = carregar_sinais_vitais()
    vitais = detectar_anomalias_sinais_vitais(df_sinais)
    presc, fontes["prescricoes"] = carregar_prescricoes()
    prescricoes = detectar_anomalias_prescricoes(presc)
    kp = resultado_video.get("keypoints_history") or []
    if len(kp) >= 2:
        fontes["movimentacao"] = "VIDEO (keypoints reais)"
    else:
        _s = os.getenv("SEMENTE", "").strip()
        cenario = (os.getenv("MOVIMENTO_CENARIO", "").strip() or "estavel")
        kp = gerar_keypoints_sinteticos(
            semente=(int(_s) if _s.isdigit() else None), cenario=cenario)
        fontes["movimentacao"] = "SINTETICO (%s)" % cenario
    movimentacao = detectar_anomalias_movimentacao(kp)
    alertas = (vitais.get("alertas", [])
               + prescricoes.get("alertas", [])
               + movimentacao.get("alertas", []))
    resumo = vitais.get("resumo", []) + prescricoes.get("resumo", [])
    resultado_sinais = {
        "fc_anomalia": vitais["fc_anomalia"],
        "bp_anomalia": vitais["bp_anomalia"],
        "spo2_anomalia": vitais["spo2_anomalia"],
        "temp_anomalia": vitais.get("temp_anomalia", False),
        "movimento_repetino": movimentacao.get("movimento_repetino", False),
        "imobilidade_anormal": movimentacao.get("imobilidade_anormal",
                                                False),
        "quedas_suspeitas": movimentacao.get("quedas_suspeitas", False),
        "paciente_estavel": (vitais.get("paciente_estavel", True)
                             and not movimentacao.get(
                                 "movimento_repetino", False)
                             and not movimentacao.get(
                                 "quedas_suspeitas", False)),
        "prescricao_estavel": prescricoes.get("estavel", False),
        "vitais": vitais, "prescricoes": prescricoes,
        "movimentacao": movimentacao, "resumo": resumo,
        "alertas": alertas, "fontes": fontes,
        "n_leituras": vitais.get("n_leituras", 0),
        "parametros": vitais.get("parametros", []),
        "n_keypoints_movimentacao": len(kp),
    }
    registrar_api("anomalias/composto",
                  {"fontes": fontes,
                   "paciente_estavel": resultado_sinais["paciente_estavel"],
                   "n_leituras": resultado_sinais["n_leituras"],
                   "n_alertas": len(alertas)})
    return resultado_sinais

resultado_sinais = executar_sinais()
print("Sinais -> paciente_estavel=%s | %d leituras (%s) | fontes=%s"
      % (resultado_sinais["paciente_estavel"],
         resultado_sinais["n_leituras"],
         ",".join(resultado_sinais["parametros"]),
         resultado_sinais["fontes"]))
for _l in resultado_sinais["resumo"]:
    print("   -", _l)
print("  anomalias:", resultado_sinais["alertas"]
      or ["nenhuma (paciente estavel)"])

## Relatório Final

Resume os resultados das três análises e produz a decisão clínica final
chamando o `workflow_integrado` com tudo que foi processado.

In [ ]:
print('=' * 70)
print('TECH CHALLENGE IADT FASE 4 - SISTEMA MULTIMODAL COMPLETO')
print('Modo de execucao:', MODO.upper())
print('=' * 70)

print('\n## 1. DEMONSTRAÇÃO DE ANÁLISE DE VÍDEO')
print('   • OpenPose-Light/MediaPipe: analise postural em {} frames'
      .format(resultado_video['frames']))
print('   • YOLOv8: {} objetos/areas criticos detectados'
      .format(resultado_video['objetos_totais']))
print('   • Anomalias posturais (score): {}'
      .format(resultado_video['anomalias_posturais']))
print('   • Alertas de video: {}'.format(resultado_video['total_alertas']))

print('\n## 2. DEMONSTRAÇÃO DE ANÁLISE DE ÁUDIO')
print('   • Transcricao: "{}"'.format(resultado_audio['transcricao'][:80]))
print('   • Sentimento: {}'
      .format(resultado_audio['sentimento'].get('rotulo', 'neutro')))
print('   • Entidades medicas: {}'
      .format(len(resultado_audio['entidades'])))
print('   • Alteracoes vocais: {}'
      .format(resultado_audio['total_alteracoes']))

print('\n## 3. DETECÇÃO DE ANOMALIAS')
print('   • Leitura de sinais vitais (fonte: {}, {} leituras)'
      .format(resultado_sinais['fontes'].get('sinais_vitais'),
              resultado_sinais.get('n_leituras', 0)))
for _l in resultado_sinais.get('resumo', []):
    print('       -', _l)
print('   • Anomalias: ' + ('; '.join(resultado_sinais.get('alertas', []))
      or 'nenhuma (paciente estavel)'))
print('   • Fontes ->', resultado_sinais.get('fontes', {}))

print('\n## 4. INTEGRAÇÃO E DECISÃO FINAL')
resultado = workflow_integrado(resultado_video, resultado_audio,
                               resultado_sinais)
print('   • Decisao:', resultado['decisao'])
print('   • Nivel:', resultado['nivel'])
print('   • Acao:', resultado['acao_recomendada'])
print('   • Alertas gerados:', resultado['total_alertas'])
print('   • Pontuacao de risco:', resultado['punteis_risco'])
print('   • Detalhe:', resultado['alertas'])

print('\n' + '=' * 70)
print('FIM DA APRESENTAÇÃO DO TECH CHALLENGE IADT FASE 4')
print('=' * 70)

In [ ]:
# =====================================================================
# RESUMO CONSOLIDADO — todos os retornos das APIs/passo (para auditoria)
# =====================================================================
registrar_api("resultado_video", resultado_video)
registrar_api("resultado_audio", resultado_audio)
registrar_api("resultado_sinais", resultado_sinais)
registrar_api("decisao_final", resultado)

print("\n" + "=" * 70)
print("LOG DE RETORNOS (APIs e etapas) — total: %d" % len(REGISTRO_API))
print("=" * 70)
for i, r in enumerate(REGISTRO_API, 1):
    print("%02d. %-30s -> %s" % (i, r["nome"], _resumir(r["dado"], 500)))
print("-" * 70)
print("Log completo tambem em:", LOG_FILE)